# Sistema de agentes para la generación y el mantenimiento de horarios académicos

Cuaderno de ejecución del proyecto. Técnicas aplicadas: problema de satisfacción de restricciones (PSR), búsqueda informada (A* ponderado) y búsqueda local con algoritmos genéticos.
Ejecutar desde la carpeta `proyecto_horarios`.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
from horarios import datos, experimentos, salida
from horarios.agentes import Orquestador

## 1. Entorno
El entorno es totalmente observable: materias, docentes, aulas, franjas y matrícula son datos conocidos antes de planificar.

In [ ]:
ent = datos.generar_entorno(20, semilla=1)
print(f"{len(ent.materias)} materias, {ent.n} sesiones, {len(ent.docentes)} docentes, {ent.na} aulas, {len(ent.grupos)} grupos")
print("Aulas:", dict(zip(ent.aulas, ent.cap)))

## 2. Planificación (PSR, algoritmo genético y búsqueda local)

In [ ]:
orq = Orquestador(ent, semilla=1)
orq.planificar(metodo_local="sa", generaciones=80)
print(salida.tabla_etapas(orq.etapas))
print("\nReferencia manual simulada:", ent.detalle(datos.horario_referencia(ent)))
print("Horario del sistema:       ", ent.detalle(orq.g))

## 3. Horario en grilla

In [ ]:
print(salida.grilla_texto(ent, orq.g, "grupo", "G1"))
print()
print(salida.grilla_texto(ent, orq.g, "docente", ent.docentes[0]))

## 4. Convergencia del algoritmo genético

In [ ]:
import matplotlib.pyplot as plt
h = orq.historia_ga
plt.figure(figsize=(7, 3.5))
plt.plot([x["gen"] for x in h], [x["mejor"] for x in h], "k-", label="Mejor")
plt.plot([x["gen"] for x in h], [x["media"] for x in h], "k--", label="Media")
plt.xlabel("Generación"); plt.ylabel("Penalización"); plt.legend(); plt.grid(color="#dddddd")
plt.show()

## 5. Cambio sorpresa: un docente deja de estar disponible
El verificador rechaza el horario y el reparador lo corrige con búsqueda informada (A* ponderado).

In [ ]:
r = orq.aplicar_evento(**orq.sugerir_sorpresa())
print(r["evento"])
print("Conflictos tras el evento:", r["conflictos_tras_evento"])
print(f"Reparación con {r['metodo']}: {r['movidas']} sesiones movidas, {r['nodos']} nodos, {r['tiempo']} s")
print("Métricas finales:", r["detalle"])
for l in orq.log[-4:]:
    print(l["t"], f"[{l['agente']}]", l["texto"])

## 6. Experimentación
Tres tamaños (pocas, medianas y muchas materias), comparación contra la referencia manual y cambio sorpresa.

In [ ]:
res = experimentos.correr(repeticiones=2, salida="resultados")
experimentos.imprimir(res)